In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub

from prompt_toolkit.input import Input

vehicle_siren_sounds_path = kagglehub.dataset_download('vishnu0399/emergency-vehicle-siren-sounds')

print('Data source import complete.')


# Introduction

## Emergency Vehicle Siren Sound Classification

### Context

This dataset was completely made manually by extracting audio from sources available on internet sites like Google and Youtube and saved as .wav audio format.

### Content

The dataset consists of wav-format audio files which are of length 3-seconds. They contain the siren sound of Emergency Vehicles - Ambulance and Firetruck. A third category named Traffic also exists where it contains 3-second .wav format audio files of plain traffic sound. Each category contains 200 sound files, 200 Spectrogram images for each audio file and a python script for converting each audio file to its spectrogram.

# Importing Libraries

In [ ]:
#Audio Processing Libraries
!pip install resampy
import resampy
import librosa
import librosa.display
import librosa.core
from scipy import signal

#For Playing Audios
import IPython.display as ipd

#Array Processing
import numpy as np

#Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Display the confusion matrix
from sklearn.metrics import confusion_matrix

#Deal with .pkl files
import pickle

#Create a dataframe
import pandas as pd

#Transform and encode the categorical targets
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

#Split dataset
from sklearn.model_selection import train_test_split

import os

# Exploring Dataset

## Ambulance

In [ ]:
filename = vehicle_siren_sounds_path +"/sounds/ambulance/sound_1.wav"
plt.figure(figsize=(14,5))
data, sample_rate = librosa.load(filename)
librosa.display.waveshow(data, sr=sample_rate)
ipd.Audio(filename)

## Firetruck

In [ ]:
filename = vehicle_siren_sounds_path +"/sounds/firetruck/sound_201.wav"
plt.figure(figsize=(14,5))
data, sample_rate = librosa.load(filename)
librosa.display.waveshow(data, sr=sample_rate)
ipd.Audio(filename)

## Traffic

In [ ]:
filename = vehicle_siren_sounds_path +"/sounds/traffic/sound_401.wav"
plt.figure(figsize=(14,5))
data, sample_rate = librosa.load(filename)
librosa.display.waveshow(data, sr=sample_rate)
ipd.Audio(filename)

# Data Preprocessing

## Using the function features_extractor to get a 80 MFCCs from each audio

In [ ]:
def features_extractor(file_name):
    audio, sample_rate = librosa.load(file_name)
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=80)
    mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)

    return mfccs_scaled_features

## Now we iterate through every audio file and extract features using Mel-Frequency Cepstral Coefficients

In [ ]:
audio_dataset_path = vehicle_siren_sounds_path +'/sounds/'

extracted_features = []
for path in os.listdir(audio_dataset_path):
    for file in os.listdir(audio_dataset_path+path+"/"):
        if file.lower().endswith(".wav"):
            file_name = audio_dataset_path+path+"/"+file
            data = features_extractor(file_name)
            extracted_features.append([data, path])

# Save the data frame into a .pkl file

In [ ]:
f = open('./Extracted_Features.pkl', 'wb')
pickle.dump(extracted_features, f)
f.close()

# Read the Extracted_Features from the .pkl file

In [ ]:
f = open('./Extracted_Features.pkl', 'rb')
Data = pickle.load(f)
f.close()

# Transform Data into a dataframe

In [ ]:
df = pd.DataFrame(Data,columns=['feature','class'])
df.head()

In [ ]:
df['class'].value_counts()

# Splitting the data into train and test sets

In [ ]:
X = np.array(df['feature'].tolist())
Y = np.array(df['class'].tolist())

In [ ]:
X.shape

In [ ]:
Y.shape

# Label Encoding

In [ ]:
labelencoder = LabelEncoder()
y = to_categorical(labelencoder.fit_transform(Y))

In [ ]:
Y[0]

In [ ]:
y[0]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y, shuffle=True)

In [ ]:
y_train.shape

# Display the shape of each splits

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

# ﷿﷿﷿﷿ Model Building

In [ ]:
!pip install scikeras
from keras.layers import *
from keras.models import *
from keras.callbacks import *
from keras import backend as K
from sklearn import metrics
from keras.callbacks import ModelCheckpoint, EarlyStopping
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
from datetime import datetime

# ﷿﷿﷿﷿ Grid Search

In [ ]:
# def display_cv_results(search_results):
#     print('Best score = {:.4f} using {}'.format(search_results.best_score_, search_results.best_params_))
#     means = search_results.cv_results_['mean_test_score']
#     stds = search_results.cv_results_['std_test_score']
#     params = search_results.cv_results_['params']
#     for mean, stdev, param in zip(means, stds, params):
#         print('mean test accuracy +/- std = {:.4f} +/- {:.4f} with: {}'.format(mean, stdev, param))

In [ ]:
# ## Traning my model
# model = cnn()
# model = KerasClassifier(build_fn=model, verbose=1)

# param_grid = {
#     'batch_size': [32, 64],
#     'epochs': [100, 200],
#     'optimizer': ['rmsprop', 'adam'],
#     'activation': ['relu'],
#     'dropout_rate': [0.0, 0.2, 0.4, 0.5]
# }

# start = datetime.now()

# grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=5)
# grid_result = grid.fit(X_train_features, y_train).history

# # print out results
# print('time for grid search = {:.0f} sec'.format(datetime.now() - start))
# display_cv_results(grid_result)

# ﷿﷿﷿﷿ CNN

In [ ]:
X_train_features  = X_train.reshape(len(X_train),-1,1)
X_test_features = X_test.reshape(len(X_test),-1,1)
print("Reshaped Array Size", X_train_features.shape)

In [ ]:
X_train.shape

In [ ]:
def cnn(optimizer="adam", activation="relu", dropout_rate=0.5):
    K.clear_session()
    inputs = Input(shape=(X_train_features.shape[1], X_train_features.shape[2]))

    #First Conv1D layer
    conv = Conv1D(3, 13, padding='same', activation=activation)(inputs)
    if dropout_rate != 0:
        conv = Dropout(dropout_rate)(conv)
    conv = MaxPooling1D(2)(conv)

    #Second Conv1D layer
    conv = Conv1D(16, 11, padding='same', activation=activation)(conv)
    if dropout_rate != 0:
        conv = Dropout(dropout_rate)(conv)
    conv = MaxPooling1D(2)(conv)

    #MaxPooling 1D
    conv = GlobalMaxPool1D()(conv)

    #Dense Layer
    conv = Dense(16, activation=activation)(conv)
    outputs = Dense(y_test.shape[1], activation='softmax')(conv)

    model = Model(inputs, outputs)
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['acc'])
    return model

In [ ]:
model_cnn = cnn(optimizer="adam", activation="relu", dropout_rate=0)
model_cnn.summary()

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model_cnn, show_shapes=True, show_layer_names=True)

In [ ]:
early_stop = EarlyStopping(monitor = 'val_accuracy', mode ='max',
                          patience = 10, restore_best_weights = True)

history = model_cnn.fit(X_train_features, y_train, epochs = 200,
                       callbacks = [early_stop],
                       batch_size = 64, validation_data = (X_test_features, y_test))

In [ ]:
# Summarize History for Loss

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Time')
plt.ylabel('Epoch')
plt.legend(['train', 'validation'], loc = 'upper left')
plt.show()

In [ ]:
_, acc = model_cnn.evaluate(X_test_features, y_test)
print("Test Accuracy : ", acc)

In [ ]:
y_pred = model_cnn.predict(X_test_features)

conf_mat = confusion_matrix(np.argmax(y_test, axis=1), np.argmax(y_pred, axis=1))
fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(conf_mat, annot=True, fmt='d', xticklabels=labelencoder.classes_, yticklabels=labelencoder.classes_, cbar=False)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
model_cnn.save('./CNN_Model.h5')

# LSTM

In [ ]:
x_train_features  = X_train.reshape(len(X_train),-1, 80)
x_test_features = X_test.reshape(len(X_test), -1, 80)
print("Reshaped Array Size", x_train_features.shape)

In [ ]:
def lstm(x_tr):
    K.clear_session()
    inputs = Input(shape=(x_tr.shape[1], x_tr.shape[2]))
    #lstm
    x = LSTM(128)(inputs)
    x = Dropout(0.5)(x)
    #dense
    x = Dense(64, activation='relu')(x)
    x = Dense(y_test.shape[1], activation='softmax')(x)
    model = Model(inputs, x)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])
    return model

In [ ]:
model_lstm = lstm(x_train_features)
model_lstm.summary()

In [ ]:
plot_model(model_lstm, show_shapes=True, show_layer_names=True)

In [ ]:
mc = ModelCheckpoint('best_model.keras', monitor='val_acc', verbose=1, save_best_only=True, mode='max')

In [ ]:
history = model_lstm.fit(x_train_features, y_train, epochs = 1000,
                        callbacks = [mc],
                        batch_size = 64, validation_data = (x_test_features, y_test))

In [ ]:
#summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.xlabel('Time')
plt.ylabel('epoch')
plt.legend(['train','validation'], loc='upper left')
plt.show()

In [ ]:
_,acc = model_lstm.evaluate(x_test_features, y_test)
print("Accuracy:", acc)

In [ ]:
y_pred = model_lstm.predict(x_test_features)

conf_mat = confusion_matrix(np.argmax(y_test, axis=1), np.argmax(y_pred, axis=1))
fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(conf_mat, annot=True, fmt='d', xticklabels=labelencoder.classes_, yticklabels=labelencoder.classes_, cbar=False)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
model_lstm.save('./LSTM.h5')

In [ ]:
print(f"X-train shape: {X_train.shape}")
print(f"x-test-features shape: {x_train_features.shape}")
print (f"X-test shape: {X_test.shape}")
print(f"x-test-features shape: {x_test_features.shape}")

# Conclusion

**In this notebook we learnt how to work with Sound Data and perform Deep Learning Methods for classification by training a CNN as well as an LSTM. And after that ResNet50**